# Code Generator

Using Open Source models to generate high performance C++ code from Python code.

In [1]:
# Imports

import os
import io
import sys

import openai
from dotenv import load_dotenv
from openai import OpenAI
import subprocess
from IPython.display import Markdown, display
import gradio as gr


In [2]:
# Using Open Source models from OLLAMA and GROQ:
load_dotenv(override= True)
ollama_api_key = ''
groq_api_key = os.getenv('GROQ_API_KEY')

ollama_base_url = 'http://172.37.23.33/11434/v1'
groq_base_url= 'https://api.groq.com/openai/v1'

ollama = OpenAI(base_url=ollama_base_url, api_key=ollama_api_key)
groq = OpenAI(base_url=groq_base_url, api_key=groq_api_key)

In [22]:
models = ['llama3.1:8b',
          'qwen2.5-coder:7b',
          'deepseek-coder-v2:16b',
          'llama-3.3-70b-versatile',
          'openai/gpt-oss-120b',
          'openai/gpt-oss-20b']

clients = [{
    'llama3.1:8b': ollama,
    'qwen2.5-coder:7b' : ollama,
    'deepseek-coder-v2:16b' : ollama,
    'llama-3.3-70b-versatile' : groq,
    'openai/gpt-oss-120b' : groq,
    'openai/gpt-oss-20b' : groq
}]

In [13]:
# System Info:

from system_info import retrieve_system_info

system_info = retrieve_system_info()
system_info

{'os': {'system': 'Windows',
  'arch': 'AMD64',
  'release': '10',
  'version': '10.0.26200',
  'kernel': '10',
  'distro': None,
  'wsl': False,
  'rosetta2_translated': False,
  'target_triple': ''},
 'package_managers': ['winget'],
 'cpu': {'brand': '13th Gen Intel(R) Core(TM) i9-13900HX',
  'cores_logical': 32,
  'cores_physical': 24,
  'simd': []},
 'toolchain': {'compilers': {'gcc': '', 'g++': '', 'clang': '', 'msvc_cl': ''},
  'build_tools': {'cmake': '', 'ninja': '', 'make': ''},
  'linkers': {'ld_lld': ''}}}

In [14]:
# Compile and Run Commands for C++ Code (from previous file):

compile_command = [ r"C:\msys64\ucrt64\bin\g++.exe", "-std=c++20", "-O3", "-march=native", "-DNDEBUG", "-flto", "main.cpp", "-o", "main.exe", "-static-libstdc++", "-static-libgcc", ]

run_command = ["main.exe"]

## Porting Code using Open Source Models:

In [15]:
# System Prompt:
system_prompt = '''
Your task is to convert Python code into high performance C++ code.
Respond only with C++ code. Do not provide any explanation other than occasional comments.
The C++ response needs to produce an identical output in the fastest possible time.
'''

# Function to create user prompt with given Python code:
def user_prompt_for(python):
    return f'''
Port this Python code to C++ with the fastest possible implementation that produces identical output in the least time.
The system information is:
{system_info}
Your response will be written to a file called main.cpp and then compiled and executed; the compilation command is:
{compile_command}
Respond only with C++ code.
Python code to port:

```python
{python}
```
'''

In [16]:
# Function to define Messages List for LLM:
def message_for(python):
    return [{'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_prompt_for(python)}]

In [17]:
# Function to write given output C++ code from LLM to a file:
def write_output(cpp):
    with open('main.cpp', 'w', encoding= 'utf-8') as f:
        f.write(cpp)

In [23]:
# Function to Port Code using OpenAI API:
def port(model, python):
    """
    Calling LLM API to with given Model to generate C++ code from given Python code.
    :param model: LLM model name
    :param python: Python Code
    """
    client = clients[model]
    response = client.chat.completions.create(model= model,
                                              messages= message_for(python),
                                              reasoning_effort= 'high')
    reply = response.choices[0].message.content
    reply = reply.replace('```cpp', '').replace('```', '')
    write_output(reply) # Writing LLMs response to file
    return reply

In [19]:
# Python code to Port to C++ (as a string):
pi = """
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(200000000, 4, 1) * 4
end_time = time.time()

print(f'Result: {result:.12f}')
print(f'Execution time: {(end_time - start_time):.8f} Seconds')
"""

In [20]:
# Function to Run Python Code:
def run_python(code):
    globals_dict = {"__builtins__": __builtins__}

    buffer = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = buffer

    try:
        exec(code, globals_dict)
        output = buffer.getvalue()
    except Exception as e:
        output = f"Error: {e}"
    finally:
        sys.stdout = old_stdout

    return output

In [21]:
# Running Python code:
run_python(code= pi)

'Result: 3.141592656089\nExecution time: 53.57323170 Seconds\n'